# Résumé de la vidéo & Code Python

## Résumé de la vidéo
Dans cette vidéo issue de la chaîne **Computer Science**, l'auteur présente une méthode complète pour prédire les prix de clôture de l'action **Apple Inc. (AAPL)** à l'aide d'un réseau de neurones récurrent de type **LSTM (Long Short-Term Memory)** sous Python (avec Keras et TensorFlow).

### Étapes principales du tutoriel :
1. **Importation des bibliothèques** : `pandas_datareader`, `numpy`, `pandas`, `scikit-learn` (MinMaxScaler), `keras` (Sequential, Dense, LSTM) et `matplotlib` [cite: 1].
2. **Récupération des données** : Téléchargement de l'historique boursier d'Apple depuis Yahoo Finance entre 2012 et fin 2019 [cite: 1].
3. **Visualisation** : Affichage de la courbe historique du prix de clôture (`Close`) [cite: 1].
4. **Préparation & Normalisation** : Extraction du prix de clôture, mise à l'échelle des données entre 0 et 1 avec `MinMaxScaler` [cite: 1].
5. **Création des ensembles d'entraînement (Training)** :
   - Utilisation d'une fenêtre glissante de **60 jours** passés pour prédire le prix du **61ème jour** [cite: 1].
   - Redimensionnement (Reshape) des tableaux NumPy en 3D (`[samples, time steps, features]`) requis par l'architecture LSTM [cite: 1].
6. **Construction & Entraînement du modèle LSTM** :
   - Première couche LSTM à 50 neurones (`return_sequences=True`) [cite: 1].
   - Seconde couche LSTM à 50 neurones (`return_sequences=False`) [cite: 1].
   - Couche dense intermédiaire à 25 neurones puis couche de sortie à 1 neurone [cite: 1].
   - Compilation avec l'optimiseur `adam` et la perte `mean_squared_error` [cite: 1].
   - Entraînement (`fit`) avec un batch size de 1 sur 1 epoch [cite: 1].
7. **Évaluation & Visualisation des prédictions** :
   - Calcul de la métrique **RMSE (Root Mean Squared Error)** [cite: 1].
   - Graphique comparatif entre les prix réels de validation et les prix prédits par le modèle [cite: 1].
8. **Test de prédiction sur une date spécifique** : Prédiction du prix de clôture pour le 18 décembre 2019 à partir des 60 jours précédents, puis comparaison avec le prix réel observé [cite: 1].

### Description du programme

In [ ]:
# Description: This program uses an artificial recurrent neural network called Long Short Term Memory (LSTM)
# to predict the closing stock price of a corporation (Apple Inc.) using the past 60 day stock price.

### 1. Importation des bibliothèques

In [ ]:
import math
import pandas_datareader as web
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from keras.models import Sequential
from keras.layers import Dense, LSTM
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')

### 2. Récupération des données boursières

In [ ]:
# Get the stock quote
df = web.DataReader('AAPL', data_source='yahoo', start='2012-01-01', end='2019-12-17')
# Show the data
df

### 3. Visualisation de l'historique du prix de clôture

In [ ]:
# Get the number of rows and columns in the data set
print(df.shape)

# Visualize the closing price history
plt.figure(figsize=(16,8))
plt.title('Close Price History')
plt.plot(df['Close'])
plt.xlabel('Date', fontsize=18)
plt.ylabel('Close Price USD ($)', fontsize=18)
plt.show()

### 4. Normalisation des données

In [ ]:
# Create a new dataframe with only the 'Close' column
data = df.filter(['Close'])
# Convert the dataframe to a numpy array
dataset = data.values
# Get the number of rows to train the model on
training_data_len = math.ceil(len(dataset) * 0.8)
print(training_data_len)

# Scale the data
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(dataset)
scaled_data

### 5. Création de l'ensemble d'entraînement

In [ ]:
# Create the training data set
# Create the scaled training data set
train_data = scaled_data[0:training_data_len, :]

# Split the data into x_train and y_train data sets
x_train = []
y_train = []

for i in range(60, len(train_data)):
    x_train.append(train_data[i-60:i, 0])
    y_train.append(train_data[i, 0])
    if i <= 60:
        print(x_train)
        print(y_train)
        print()

# Convert the x_train and y_train to numpy arrays
x_train, y_train = np.array(x_train), np.array(y_train)

# Reshape the data
x_train = np.reshape(x_train, (x_train.shape[0], x_train.shape[1], 1))
print(x_train.shape)

### 6. Création et entraînement du modèle LSTM

In [ ]:
# Build the LSTM model
model = Sequential()
model.add(LSTM(50, return_sequences=True, input_shape=(x_train.shape[1], 1)))
model.add(LSTM(50, return_sequences=False))
model.add(Dense(25))
model.add(Dense(1))

# Compile the model
model.compile(optimizer='adam', loss='mean_squared_error')

# Train the model
model.fit(x_train, y_train, batch_size=1, epochs=1)

### 7. Préparation des données de test

In [ ]:
# Create the testing data set
# Create a new array containing scaled values from index 1543 to 2003
test_data = scaled_data[training_data_len - 60:, :]

# Create the data sets x_test and y_test
x_test = []
y_test = dataset[training_data_len:, :]

for i in range(60, len(test_data)):
    x_test.append(test_data[i-60:i, 0])

# Convert the data to a numpy array
x_test = np.array(x_test)

# Reshape the data
x_test = np.reshape(x_test, (x_test.shape[0], x_test.shape[1], 1))

### 8. Prédiction et Évaluation (RMSE)

In [ ]:
# Get the models predicted price values
predictions = model.predict(x_test)
predictions = scaler.inverse_transform(predictions)

# Get the root mean squared error (RMSE)
rmse = np.sqrt(np.mean((predictions - y_test)**2))
print('RMSE:', rmse)

### 9. Visualisation des résultats de prédiction

In [ ]:
# Plot the data
train = data[:training_data_len]
valid = data[training_data_len:]
valid['Predictions'] = predictions

# Visualize the data
plt.figure(figsize=(16,8))
plt.title('Model')
plt.xlabel('Date', fontsize=18)
plt.ylabel('Close Price USD ($)', fontsize=18)
plt.plot(train['Close'])
plt.plot(valid[['Close', 'Predictions']])
plt.legend(['Train', 'Val', 'Predictions'], loc='lower right')
plt.show()

# Show the valid and predicted prices
print(valid)

### 10. Prédiction pour une journée spécifique (ex: 18 Décembre 2019)

In [ ]:
# Get the quote
apple_quote = web.DataReader('AAPL', data_source='yahoo', start='2012-01-01', end='2019-12-17')
# Create a new dataframe
new_df = apple_quote.filter(['Close'])
# Get the last 60 day closing price values and convert the dataframe to an array
last_60_days = new_df[-60:].values
# Scale the data to be values between 0 and 1
last_60_days_scaled = scaler.transform(last_60_days)

# Create an empty list
X_test = []
# Append the past 60 days
X_test.append(last_60_days_scaled)
# Convert the X_test data set to a numpy array
X_test = np.array(X_test)
# Reshape the data
X_test = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], 1))

# Get the predicted scaled price
pred_price = model.predict(X_test)
# Undo the scaling
pred_price = scaler.inverse_transform(pred_price)
print('Predicted Price:', pred_price)

# Get the actual price for 2019-12-18
apple_quote2 = web.DataReader('AAPL', data_source='yahoo', start='2019-12-18', end='2019-12-18')
print('Actual Price:', apple_quote2['Close'])